# Layer normalization

### Definition

Input
* $x \in \mathbb{R}^{d_{in}}$ 

Weights
* scale $\gamma \in \mathbb{R}^{d_{in}}$ 
* shift $\beta \in \mathbb{R}^{d_{in}}$

Output
* $o \in \mathbb{R}^{d_{in}}$

$$o = \text{LayerNorm}_{\gamma,\beta}(x)= \gamma * \frac{x - \mu(x)}{\sqrt{\sigma(x)^2 + \varepsilon}} + \beta.$$
where
  $$
  \mu(x) = \frac{1}{d_{in}}\sum_{i=1}^{d_{in}} x_{i}, 
  \qquad
  \sigma(x)^2 = \frac{1}{d_{in}}\sum_{i=1}^{d_{in}} (x_{i} - \mu(x))^2.
  $$


where $*$ denotas the elementwise vector-multiplication and $\epsilon >0$ is a small quantity for avoiding zero division. 

## Code

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import Tensor, nn
from torch.utils.data import DataLoader, TensorDataset

import models.deep_learning.components as mynn


## Testing

### Weights

In [ ]:
layernorm = mynn.LayerNorm(normalized_shape=(3, 5))
nn_layernorm = nn.LayerNorm(normalized_shape=[3, 5])
print("nn layernorm")
for name, w in layernorm.named_parameters():
    print(f"{name} {w}")
print("\ncustom layernorm")
for name, w in nn_layernorm.named_parameters():
    print(f"{name} {w}")

### output

In [ ]:
x = torch.randn(3, 4, 2)
layernorm = mynn.LayerNorm(normalized_shape=(4, 2))
nn_layernorm = nn.LayerNorm(normalized_shape=[4, 2])
output = layernorm(x)
output_nn = nn_layernorm(x)
print(output)
print(output_nn)

### Output (stacked layernorms)
Notice that the custom implementation differ slightly in numerical values due to floating-point accumulation order

In [ ]:
class MLPLN(nn.Module):
    def __init__(
        self,
        ln_net_cls: type[nn.LayerNorm] | type[mynn.LayerNorm],
        input_dim: int,
        hidden_dim: list[int],
        output_dim: int,
        pos: bool = False,
    ):
        super().__init__()
        layers: list[nn.Module] = [nn.Flatten()]
        in_dim = input_dim
        for h_dim in hidden_dim:
            if pos:
                layers.append(
                    nn.Linear(in_dim, h_dim, bias=False)
                )  # bias learned by LinearLayer
                if in_dim > 1:
                    layers.append(ln_net_cls(h_dim))
            else:
                # Only pass elementwise_affine if using nn.LayerNorm
                if in_dim > 1:
                    layers.append(ln_net_cls(in_dim, elementwise_affine=False))
                layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.ReLU())
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, output_dim, bias=False))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

In [ ]:
x = torch.randn(5, 28, 28)
torch.manual_seed(1)
out_refs: list[torch.Tensor] = []
for i in range(10):
    net = MLPLN(nn.LayerNorm, input_dim=28 * 28, hidden_dim=i * [512], output_dim=100)
    out = net(x)
    out_norm = out.norm().item()
    print(f"{i} - {out_norm = }")
    out_refs.append(out)


In [ ]:
torch.manual_seed(1)
for i in range(10):
    net = MLPLN(mynn.LayerNorm, input_dim=28 * 28, hidden_dim=i * [512], output_dim=100)
    out = net(x)
    out_norm = out.norm().item()
    print(
        f"{i} - {out_norm = } - allclose: {torch.allclose(out_refs[i], out, atol=1e-5, rtol=0.0)}"
    )

### Train 

#### generate data

In [ ]:
def f(x: Tensor) -> Tensor:
    return 1 + 2 * x**2


torch.manual_seed(3)
N = 32
xs = (4 * torch.rand(N) - 2)[:, None]
ys = f(xs) + 0.5 * torch.randn(N, 1)
x_eval = torch.linspace(-2, 2, 100)

dataset = TensorDataset(xs, ys)

In [ ]:
plt.plot(xs[:, 0].detach().numpy(), ys[:, 0].detach().numpy(), "o")
plt.plot(x_eval.detach().numpy(), f(x_eval).detach().numpy(), "--")
plt.grid()
plt.show()

In [ ]:
lr = 0.001
momentum = 0.9
batch_size = N // 4
hidden_dim = [64, 64]
loss_fn = nn.MSELoss()
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

### trainning

##### Torch LayerNorm

In [ ]:
torch.manual_seed(1)

nn_model = MLPLN(nn.LayerNorm, input_dim=1, hidden_dim=hidden_dim, output_dim=1)
opt = torch.optim.SGD(nn_model.parameters(), lr=lr, momentum=momentum)

nn_model.train()
torch_losses = []
for epoch in range(100):
    for x, y in data_loader:
        preds = nn_model(x)
        loss = loss_fn(preds, y)
        loss.backward()
        opt.step()
        opt.zero_grad()
        torch_losses.append(loss.item())
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

##### Custom LayerNorm

In [ ]:
torch.manual_seed(1)
model = MLPLN(mynn.LayerNorm, input_dim=1, hidden_dim=hidden_dim, output_dim=1)
opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum)
model.train()
my_losses = []

for epoch in range(100):
    for x, y in data_loader:
        preds = model(x)
        loss = loss_fn(preds, y)
        loss.backward()
        opt.step()
        opt.zero_grad()
        my_losses.append(loss.item())
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

In [ ]:
plt.plot(torch_losses, linestyle="-", label="nn.BNPreLoss")
plt.plot(my_losses, linestyle="--", label="BNPreLoss")
plt.title("Torch BatchNorm1d vs My BatchNorm1d")
plt.grid(True)
plt.legend()
plt.show()

### Evaluation

In [ ]:
nn_model.eval()
model.eval()

plt.plot(
    x_eval,
    nn_model(x_eval[:, None]).squeeze().detach().numpy(),
    "-",
    label="torch_pred",
)
plt.plot(
    x_eval, model(x_eval[:, None]).squeeze().detach().numpy(), "--", label="my_pred"
)
plt.plot(x_eval, f(x_eval), "--", label="true")
plt.plot(xs[:, 0].detach().numpy(), ys[:, 0].detach().numpy(), "o", label="data")
plt.legend()
plt.grid()
plt.show()